In [ ]:
# Accuracy of Model

> Accuracy of models and copy the failed images in a place

In [ ]:
#| default_exp pass_fail_image_copy

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import numpy as np
import pandas as pd
from pathlib import Path
import logging
from tqdm.auto import tqdm
from fastcore.all import *
import shutil
from argparse import ArgumentParser

In [ ]:
#| export
path=Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/nbs')
custom_lib_path=Path(r'/home/ai_warstein/custom_libs/goni/custom_libs')
sys.path.append(str(custom_lib_path))
from dotenv import load_dotenv
dotenv_p = load_dotenv(Path(Path(path).parent, 'private_easy_pin_detection/.env'))
CV_TOOLS = os.getenv('CV_TOOLS')
PRIVATE_EASY_PIN_DETECTION = os.getenv('PRIVATE_EASY_PIN_DETECTION')
sys.path.append(CV_TOOLS)
sys.path.append(PRIVATE_EASY_PIN_DETECTION)

In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| exporti
from private_easy_pin_detection.core import *
from private_easy_pin_detection.pin_map import *

In [ ]:
#| export
def get_logger(file_name):
    # Create a logger object
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)
    
    # Create a file handler
    file_handler = logging.FileHandler(file_name)
    file_handler.setLevel(logging.DEBUG)
    
    # Create a formatter
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    
    # Add the file handler to the logger
    logger.addHandler(file_handler)
    
    return logger

In [ ]:
#| hide
im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped')
msk_path=Path(r'M:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\missing_3109\missing_3109_unzip\main_im2_cropped_masks\v5')
#msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/im2_test_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5')
msk_path.ls()

In [ ]:
#| export
def create_df(mask_path):
    df = pd.DataFrame()
    
    len_ = len(Path(mask_path).ls(file_exts='.png'))
    try:
        print(f' number of files found = {len_}')
        for idx, i in tqdm(enumerate(Path(mask_path).ls(file_exts='.png')), 
                            total=len_, desc='creating df'):
            df.loc[idx, 'rec_name'] = get_m_name(f'{i}')
            df.loc[idx, 'file_path'] = f'{i}'
            df.loc[idx, 'fn']  = Path(i).name
            df.loc[idx, 'act_pc'] = pin_count_from_fn(f'{i}')


    
        not_pin = [i for i in set(df['rec_name'].values) if i not in pin_count_dict().keys()]
        if not not_pin:
            print("\033[92m" + f'Number of recipe not found = {not_pin}' + "\033[0m")  # Print in red
        else:
            print("\033[91m" + f'Number of recipe not found = {not_pin}' + "\033[0m")  # Print in green
        df['expected_pc'] = df['rec_name'].map(pin_count_dict())
        df['diff'] = df['act_pc'] - df['expected_pc']
    except Exception as e:
        print(f'Error in create_df = {e}')
    return df


In [ ]:
#| hide
df = create_df(msk_path)

In [ ]:
#| export
def get_add_and_mis_fns(df:pd.DataFrame)->tuple[np.array, np.array, np.array]:
    'Getting additional and missing pins and passsed files separate'
    failed_images_df = df.query('diff != 0')
    print(f'Number of failed images = {len(failed_images_df)}')
    missing_files = list(failed_images_df.query('diff <0')['file_path'].values)
    additional_pins = list(failed_images_df.query('diff >0')['file_path'].values)
    passed_images = list(df.query('diff == 0')['file_path'].values)

    return missing_files, additional_pins, passed_images

In [ ]:
missing_df, a_df, p_df = get_add_and_mis_fns(df)

In [ ]:
missing_df

In [ ]:
#| export
def create_folder(
        folder_name, # full path
        )->None:
    'When folder name is given, then image, mask and overlay folders are created'
    folder_im_path = Path(folder_name, 'images')
    folder_msk_path = Path(folder_name, 'masks')
    folder_overlay_path = Path(folder_name, 'overlay')
    folder_im_path.mkdir(parents=True, exist_ok=True)
    folder_msk_path.mkdir(parents=True, exist_ok=True)
    folder_overlay_path.mkdir(parents=True, exist_ok=True)

In [ ]:
#| export
def move_images(
        name:str, # name of the image file
        im_src:Path,
        msk_src:Path,
        pr_im_src:Path,
        im_dst:Path,
        msk_dst:Path,
        pr_im_dst:Path,
    ):
    'Copy images, masks and overlay images to the destination folders'
    shutil.copyfile(
        f'{im_src}/{name}', 
        f'{im_dst}/{name}')
    shutil.move(
        f'{msk_src}/{name}', 
        f'{msk_dst}/{name}')
    shutil.move(
        f'{pr_im_src}/{name}',
         f'{pr_im_dst}/{name}')


In [ ]:
#| hide
pr_im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks/overlay_masks_time_18_17_57_val_frGrnd0.9570_epoch_180.h5')

In [ ]:
#| export
def move_(
    im_path:str,
    msk_path:str,
    overlay_msk_path:str,
        path:str,# whether missing, additional or passed
        file_list:list[str],
        case='missing'
        )->None:
    'If file exist then copy images, masks and overlay images to the destination folders'
    if len(file_list) >=1:
        if case in ['missing', 'additional']:
            create_folder(f'{path}/failed/{case}')
            new_path = Path(f'{path}/failed/{case}')
            for i in tqdm(file_list, total=len(file_list)):
                name_ = Path(i).name
        
                move_images(
                    im_src=im_path,
                    msk_src=msk_path,
                    pr_im_src=overlay_msk_path,
                    name=name_, 
                    im_dst=Path(new_path, 'images'),
                    msk_dst=Path(new_path, 'masks'),
                    pr_im_dst=Path(new_path, 'overlay')
                    )
        else:
            create_folder(f'{path}/passed')
            for i in tqdm(file_list, total=len(file_list)):
                name_ = Path(i).name
                move_images(
                    im_src=im_path,
                    msk_src=msk_path,
                    pr_im_src=overlay_msk_path,
                    name=name_, 
                    im_dst=Path(f'{path}/passed/images'),
                    msk_dst=Path(f'{path}/passed/masks'),
                    pr_im_dst=Path(f'{path}/passed/overlay')
                    )
   

In [ ]:
#| hide
move_(
    path=msk_path,
    file_list=a_df,
    case='additional'
)

In [ ]:
#| hide
move_(
    path=msk_path,
    file_list=missing_df,
    case='missing'
)

In [ ]:
#| hide
move_(
    path=msk_path,
    file_list=p_df,
    case='passed'
)

In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser()
    parser.add_argument(
        '-m', '--mask_path',
        help='Path to the mask folder',
        required=True
    )
    parser.add_argument(
        '-i', '--im_path',
        help='Path to the image folder',
        required=True
    )
    parser.add_argument(
        '-o', '--overlay_path',
        help='Path to the overlay image folder',
        required=True
    )
    parser.add_argument(
        '-l', '--log_file'
        )
    return parser.parse_args()

In [ ]:
#| export
def main_():
    args = parse_args_() 
    im_path = Path(args.im_path)
    msk_path = Path(args.mask_path)
    overlay_path = Path(args.overlay_path)

    move_images_ = partial(
        move_images,
        im_src=im_path,
        msk_src=msk_path,
        pr_im_src=overlay_path,
    )
    logger = get_logger(args.log_file)

    logger.info('Started creating dataframe')
    df = create_df(msk_path)
    logger.info('Dataframe creation is done')
    logger.info('Started getting missing and additional files')
    print(f'{"#"*50}')
    print(f'{"*"*50}')
    logger.info('Started creating failed and passed iamges')

    # getting missing, addional and passed images
    missing_images, a_images, p_images = get_add_and_mis_fns(df)
    logger.info('Started creating failed and passed iamges')
    print(f'{"#"*50}')

    print(f'{"*"*50}')
    logger.info(f'Number of addtional pins ={len(a_images)}')
    logger.info(f'Number of missing pins ={len(missing_images)}')
    logger.info(f'Number of passed pins ={len(p_images)}')
    logger.info('Working on addtional images')
    move_(
        im_path=im_path,
        msk_path=msk_path,
        overlay_msk_path=overlay_path,
        path=msk_path,
        file_list=a_images,
        case='additional'
    )

    logger.info('Working on Missing pin images')
    move_(
        im_path=im_path,
        msk_path=msk_path,
        overlay_msk_path=overlay_path,
        path=msk_path,
        file_list=missing_images,
        case='missing'
    )
    logger.info('Working on images')
    move_(
        im_path=im_path,
        msk_path=msk_path,
        overlay_msk_path=overlay_path,
        path=msk_path,
        file_list=p_images,
        case='passed'
    )
    print(f'{"#"*50} Done')

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('19_model_accuracy_and_copy_failed_image.ipynb')